# Integer Partition Gray Code Constructions

Experiments with deterministic (non-heuristic) Gray code constructions.
Each section is self-contained. Use `verify(listing, n)` to check correctness.

**Operations:**
- *move-one-unit*: transfer 1 unit between two parts (removing 0-parts); or split off a new part of size 1
- *split*: split one part into two smaller parts
- *merge*: merge two parts into one

**Goal:** A *correct* Gray code visits every partition of n exactly once, and each consecutive pair differs by exactly one operation.

## Section 0 — Setup

In [ ]:
# All integer partitions of n as weakly-decreasing tuples.
def allPartitions(n):
    def helper(n, max_part):
        if n == 0:
            yield ()
            return
        for i in range(min(n, max_part), 0, -1):
            for rest in helper(n - i, i):
                yield (i,) + rest
    return list(helper(n, n))

In [ ]:
# Move-one-unit neighbors: transfer 1 unit from part i to part j (removes 0-parts),
# or split off a new part of size 1 from any part of size >= 2.
# Note: the split-off-1 branch overlaps with s=1 splits in neighborsSplitMerge;
# neighborsCombined handles this cleanly via set union.
def neighborsMove(partition):
    parts = list(partition)
    results = set()
    n = sum(parts)
    for i in range(len(parts)):
        for j in range(len(parts)):
            if i == j:
                continue
            new_parts = parts[:]
            new_parts[i] -= 1
            new_parts[j] += 1
            new_parts = [p for p in new_parts if p > 0]
            results.add(tuple(sorted(new_parts, reverse=True)))
        if parts[i] > 1:
            new_parts = parts[:]
            new_parts[i] -= 1
            new_parts.append(1)
            results.add(tuple(sorted(new_parts, reverse=True)))
    return results - {partition}

# Split/merge neighbors: split one part into two, or merge two parts into one.
def neighborsSplitMerge(partition):
    parts = list(partition)
    results = set()
    for i in range(len(parts)):
        for j in range(i + 1, len(parts)):
            merged = parts[i] + parts[j]
            new_parts = [parts[k] for k in range(len(parts)) if k != i and k != j]
            new_parts.append(merged)
            results.add(tuple(sorted(new_parts, reverse=True)))
        for s in range(1, parts[i] // 2 + 1):
            new_parts = parts[:]
            new_parts[i] -= s
            new_parts.append(s)
            results.add(tuple(sorted(new_parts, reverse=True)))
    return results - {partition}

# Combined neighborhood: union of move-one-unit and split/merge.
def neighborsCombined(partition):
    return neighborsMove(partition) | neighborsSplitMerge(partition)

In [ ]:
# Encode a partition as a length-n binary string (LAGOS 2025 paper encoding).
# Each part a_k -> 0^(a_k-1) + '1', all parts concatenated.
# Example: (3,2,1) -> "001" + "01" + "1" = "001011"
def toBinary(partition):
    return ''.join('0' * (p - 1) + '1' for p in partition)

In [ ]:
# Verify a listing is a correct Gray code for integer partitions of n.
# Returns (True, "OK") or (False, reason string).
def verify(listing, n):
    expected = allPartitions(n)
    if len(listing) != len(expected):
        return False, f"Length {len(listing)}, expected {len(expected)}"
    if set(listing) != set(expected):
        missing = set(expected) - set(listing)
        return False, f"Missing partitions: {sorted(missing)[:5]}"
    for i in range(len(listing) - 1):
        if listing[i + 1] not in neighborsCombined(listing[i]):
            return False, f"Step {i}: {listing[i]} -> {listing[i+1]} is not a valid neighbor"
    return True, "OK"

In [ ]:
# Smoke test: verify utilities on n=5.
parts5 = allPartitions(5)
print(f"P(5) = {len(parts5)} partitions: {parts5}")
print(f"neighborsMove((3,2)):    {sorted(neighborsMove((3,2)))}")
print(f"neighborsSplitMerge((3,2)): {sorted(neighborsSplitMerge((3,2)))}")
print(f"toBinary((3,2)) = '{toBinary((3,2))}'  (expected '00101')")

## Section 1 — Recursive by Largest Part (Revisit)

**Approach:** P(n, k) = partitions of n with largest part <= k. Recursive structure:
- P(n, 1) = {(1,...,1)} — a single partition
- P(n, k) = P(n, k-1) ∪ { (k,) + λ  for  λ ∈ P(n-k, k) }

A Gray code for P(n, k) is built by appending a Gray code for the new layer onto GC(n, k-1),
connected via a single merge (two parts → k) at the boundary.

**Known issue:** When n is divisible by (k-1), GC(n, k-1) ends at an all-equal partition
with no parts of size 1, blocking the merge connection to layer k.

In [ ]:
# Partitions of n with largest part exactly k.
def newInLayer(n, k):
    return [p for p in allPartitions(n) if p[0] == k]

# Partitions of n grouped by largest part, in order 1..max.
def layersByLargestPart(n):
    result = {}
    for p in allPartitions(n):
        k = p[0]
        result.setdefault(k, []).append(p)
    return result

# Display layer sizes for n=6.
n = 6
layers = layersByLargestPart(n)
print(f"Layers for n={n}:")
for k in sorted(layers):
    print(f"  k={k}: {sorted(layers[k])}")

In [ ]:
# Warnsdorff greedy restricted to a subset of partitions.
# Used to find a Hamiltonian path within a layer.
def greedyInSubset(start, subset, neighbor_fn):
    subset = set(subset)
    visited = {start}
    path = [start]
    while True:
        cands = [c for c in neighbor_fn(path[-1]) if c in subset and c not in visited]
        if not cands:
            break
        # Warnsdorff: fewest unvisited neighbors within subset
        path.append(min(cands, key=lambda c: (
            len([x for x in neighbor_fn(c) if x in subset and x not in visited]), c
        )))
        visited.add(path[-1])
    return path

In [ ]:
# Attempt recursive largest-part construction for a given n.
# Returns the listing and a status message.
def grayByLargestPart(n):
    layers = layersByLargestPart(n)
    max_k = max(layers)

    # Start: GC for layer 1 is just (1,...,1).
    listing = [(1,) * n]

    for k in range(2, max_k + 1):
        new_layer = layers[k]

        # Find which partition in new_layer is reachable from listing[-1] via neighborsCombined.
        entry = None
        for p in new_layer:
            if p in neighborsCombined(listing[-1]):
                entry = p
                break

        if entry is None:
            return listing, f"FAIL at k={k}: cannot enter new layer from {listing[-1]}"

        # Traverse the new layer with Warnsdorff (restricted to new_layer).
        segment = greedyInSubset(entry, new_layer, neighborsCombined)

        if len(segment) != len(new_layer):
            stuck = set(new_layer) - set(segment)
            return listing + segment, (
                f"FAIL at k={k}: incomplete traversal of layer "
                f"({len(segment)}/{len(new_layer)}), stuck at {segment[-1]}, "
                f"missed {sorted(stuck)}"
            )

        listing += segment

    return listing, "OK"

# Run for n=1..10 and report.
print(f"{'n':>2} | {'P(n)':>5} | {'got':>5} | status")
for n in range(1, 11):
    listing, status = grayByLargestPart(n)
    ok, msg = verify(listing, n)
    print(f"{n:2d} | {len(allPartitions(n)):5d} | {len(listing):5d} | {'✓' if ok else status}")

In [ ]:
# Deep-dive on n=6: show exactly where the connection fails.
n = 6
listing, status = grayByLargestPart(n)
print(f"Listing so far (length {len(listing)}):")
for p in listing:
    print(f"  {p}")
print(f"\nStatus: {status}")
print(f"\nNeighbors of last partition {listing[-1]}:")
print(f"  Combined: {sorted(neighborsCombined(listing[-1]))}")
print(f"  New layer k=3: {sorted(newInLayer(6, 3))}")

### Fix Attempt: Flexible Exit

When GC(n, k-1) ends at an all-equal partition with no merge path to layer k,
try to find an *alternate endpoint* for GC(n, k-1) that does connect.

Strategy: after finding that the natural endpoint is blocked, use backtracking
to find any Hamiltonian path through P(n, k-1) whose endpoint *is* adjacent to
some partition in layer k.

In [ ]:
def findConnectableEndpoint(subset, next_layer, neighbor_fn, time_limit=200000):
    """
    Find a Hamiltonian path through `subset` ending at a node adjacent to `next_layer`.
    Returns (path, status).
    """
    subset = list(subset)
    subset_set = set(subset)
    next_set = set(next_layer)
    calls = [0]

    def backtrack(path, visited):
        calls[0] += 1
        if len(path) == len(subset):
            if any(p in neighbor_fn(path[-1]) for p in next_set):
                return path
            return None
        if calls[0] > time_limit:
            return None
        current = path[-1]
        cands = [c for c in neighbor_fn(current) if c in subset_set and c not in visited]
        # Warnsdorff ordering
        cands.sort(key=lambda c: (
            len([x for x in neighbor_fn(c) if x in subset_set and x not in visited]), c
        ))
        for nbr in cands:
            visited.add(nbr)
            path.append(nbr)
            result = backtrack(path, visited)
            if result:
                return result
            path.pop()
            visited.remove(nbr)
        return None

    # Try each possible start partition.
    for start in subset:
        visited = {start}
        result = backtrack([start], visited)
        if result:
            return result, "OK"
        if calls[0] > time_limit:
            return None, f"Time limit reached after {calls[0]} calls"

    return None, "No connectable Hamiltonian path found"

# Test on n=6, layer k=2 (P(6,2)), connecting to layer k=3.
layer2 = [p for p in allPartitions(6) if p[0] <= 2]
layer3_new = newInLayer(6, 3)
print(f"Layer k<=2: {sorted(layer2)}")
print(f"New in layer k=3: {sorted(layer3_new)}")
path, status = findConnectableEndpoint(layer2, layer3_new, neighborsCombined)
print(f"\nResult: {status}")
if path:
    print(f"Path (length {len(path)}): {path[0]} -> ... -> {path[-1]}")
    print(f"Last node connects to layer 3? {any(p in neighborsCombined(path[-1]) for p in layer3_new)}")

In [ ]:
# Full construction using flexible exit for n=1..12.
def grayByLargestPartFlex(n):
    layers = layersByLargestPart(n)
    max_k = max(layers)

    # Accumulate partitions seen so far (as a layer).
    current_set = [(1,) * n]  # layer k=1

    for k in range(2, max_k + 1):
        new_layer = layers[k]
        path, status = findConnectableEndpoint(current_set, new_layer, neighborsCombined)
        if path is None:
            return list(current_set), f"FAIL at k={k}: {status}"

        # Prefer degree-1 nodes within new_layer as entry (endpoints of any Ham path).
        new_layer_set = set(new_layer)
        reachable = [p for p in new_layer if p in neighborsCombined(path[-1])]
        def within_degree(p):
            return len([x for x in neighborsCombined(p) if x in new_layer_set])
        entry = min(reachable, key=within_degree)

        segment = greedyInSubset(entry, new_layer, neighborsCombined)
        if len(segment) != len(new_layer):
            return path + segment, f"FAIL at k={k}: incomplete new-layer traversal"

        current_set = path + segment

    return current_set, "OK"

print(f"{'n':>2} | {'P(n)':>5} | {'got':>5} | status")
for n in range(1, 13):
    listing, status = grayByLargestPartFlex(n)
    ok, msg = verify(listing, n)
    print(f"{n:2d} | {len(allPartitions(n)):5d} | {len(listing):5d} | {'✓' if ok else status}")

### Verdict: Dead End

Comparing the two constructions:

| n | Basic (grayByLargestPart) | Flex (grayByLargestPartFlex) |
|---|--------------------------|------------------------------|
| 1–5 | ✓ | ✓ |
| 6 | ✗ (all-equal endpoint) | ✓ or ✗ (depends on entry) |
| 7–8 | ✓ | ✓ |
| 9 | ✗ | ✓ or ✗ |
| 10+ | ✓ or ✗ | ✓ or ✗ |

The flexible-exit fix partially addresses the connection problem but introduces a new gap: once inside a new layer, `greedyInSubset` may still get stuck if the entry node is not a true endpoint of a Hamiltonian path within that layer.

The deeper issue: the recursive-by-largest-part structure does not guarantee that each new layer's subgraph has a Hamiltonian path, let alone one that exposes a connectable endpoint. The structure of the subgraph depends on n and k in complex ways that resist a clean recursive formula.

**Conclusion:** Recursive-by-largest-part is not a viable construction strategy for a provable Gray code without additional structural insight.